In [1]:
import torch
import librosa
import laion_clap
import numpy as np
from transformers import AutoProcessor, ClapModel

/home/eduard/anaconda3/envs/music-gen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load CLAP model (music-optimized checkpoint)
model = laion_clap.CLAP_Module(enable_fusion=False, amodel='HTSAT-base')
model.load_ckpt('ckpts/music_audioset_epoch_15_esc_90.14.pt')  # music-specialized

/home/eduard/anaconda3/envs/music-gen/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Loading weights: 100%|██████████████████████████████████████████████████| 197/197 [00:00<00:00, 1915.25it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect iden

Load the specified checkpoint ckpts/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0

In [4]:
# Load audio (48kHz required)
audio, sr = librosa.load('samples/reference.wav', sr=48000)
audio = audio.reshape(1, -1)  # (1, T) format

In [6]:
# Get audio embedding (512-dim vector)
audio_embed = model.get_audio_embedding_from_data(x=audio, use_tensor=False) # must be True, but there is version conflict

In [7]:
audio_embed

array([[-0.027179  ,  0.00922796, -0.10100164, -0.06320574,  0.09926998,
        -0.00633026, -0.11198632, -0.03113993,  0.07305995, -0.00056155,
        -0.06189312,  0.02279475, -0.00716564,  0.04767765,  0.00253415,
        -0.09341485,  0.03411582,  0.0525968 ,  0.10065361,  0.00648874,
        -0.00650092,  0.0125693 , -0.0134824 ,  0.00800462,  0.01109395,
        -0.01142022, -0.01985238,  0.0016941 ,  0.02145506, -0.01212153,
         0.01762712,  0.05082301,  0.06784652,  0.02910076,  0.08162691,
         0.01323452,  0.01985021, -0.05276508, -0.00947716, -0.02508342,
        -0.04711724,  0.00649087,  0.00391038,  0.0302629 ,  0.08327347,
        -0.05120363, -0.05182417,  0.03398146, -0.0059022 ,  0.00221691,
        -0.0289368 , -0.01279024, -0.02190701,  0.02594923, -0.03144644,
         0.04474296,  0.03014815,  0.021832  ,  0.04757285, -0.01242196,
         0.01089231, -0.03179226, -0.01185306,  0.02109783, -0.03950517,
         0.02951333, -0.04207869, -0.04126921, -0.0

In [16]:
audio_embed = torch.from_numpy(audio_embed)

In [8]:
def extract_rhythmic_features(audio_path, sr=22050):
    y, _ = librosa.load(audio_path, sr=sr)
    return {
        'tempo': librosa.beat.beat_track(y=y, sr=sr)[0],
        'energy': np.mean(librosa.feature.rms(y=y)),
        'spectral_centroid': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
        'zero_crossing_rate': np.mean(librosa.feature.zero_crossing_rate(y=y)),
        'chroma_mean': np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1).tolist()
    }

In [10]:
rhythmic_features = extract_rhythmic_features('samples/reference.wav')

In [19]:
rhythmic_features

{'tempo': array([99.38401442]),
 'energy': np.float32(0.2567573),
 'spectral_centroid': np.float64(1224.1556494277565),
 'zero_crossing_rate': np.float64(0.032119311292325495),
 'chroma_mean': [0.7404283881187439,
  0.5348787903785706,
  0.3898921012878418,
  0.39737585186958313,
  0.3787176311016083,
  0.33275336027145386,
  0.35260096192359924,
  0.5096715688705444,
  0.4781860113143921,
  0.41309237480163574,
  0.43644994497299194,
  0.5180619359016418]}

In [11]:
# Precompute text embeddings for a controlled vocabulary
TAG_VOCAB = [
    "lo-fi hip hop", "drum and bass", "cinematic orchestral", 
    "analog synth", "acoustic guitar", "808 drum machine",
    "reverb-heavy vocals", "sidechain compression", "modular sequencing"
]

text_embeds = model.get_text_embedding(TAG_VOCAB, use_tensor=True)

In [15]:
text_embeds

tensor([[-0.0311, -0.0610,  0.0187,  ..., -0.0328,  0.0545, -0.0341],
        [-0.0049, -0.0448, -0.0513,  ...,  0.0167, -0.0625,  0.0393],
        [-0.0696, -0.0663, -0.0468,  ...,  0.0190,  0.0228,  0.0368],
        ...,
        [-0.0923,  0.0217, -0.0744,  ...,  0.0011, -0.0215,  0.0529],
        [-0.0810,  0.0596, -0.0417,  ...,  0.0025, -0.0450,  0.0957],
        [-0.0972, -0.0101, -0.0147,  ...,  0.0030, -0.1001,  0.0335]],
       grad_fn=<DivBackward0>)

In [12]:
# Find closest tags to your audio
def get_semantic_tags(audio_embed, text_embeds, tags, top_k=5):
    similarities = torch.nn.functional.cosine_similarity(
        audio_embed, text_embeds, dim=1
    )
    top_idx = torch.topk(similarities, top_k).indices
    return [(tags[i], similarities[i].item()) for i in top_idx]

In [20]:
semantic_tags = get_semantic_tags(audio_embed, text_embeds, TAG_VOCAB)
# → [('analog synth', 0.87), ('sidechain compression', 0.82), ...]

In [21]:
semantic_tags

[('lo-fi hip hop', 0.275363028049469),
 ('808 drum machine', 0.24586784839630127),
 ('sidechain compression', 0.24460382759571075),
 ('drum and bass', 0.15425853431224823),
 ('reverb-heavy vocals', 0.12697862088680267)]

In [30]:
def generate_suno_prompt(semantic_tags, rhythmic_features):
    tempo = rhythmic_features['tempo'].item()
    energy = rhythmic_features['energy'].item()
    spectral_centroid = rhythmic_features['spectral_centroid'].item()
    # Pseudocode for LLM augmentation
    return f'''
    You are a music prompt engineer for Suno AI. 
    Given these extracted features from a reference track:
    - Semantic tags: {semantic_tags}
    - Tempo: {tempo} BPM
    - Energy level: {energy}/1.0
    - Spectral character: {spectral_centroid}
    
    Generate a concise, effective Suno prompt (max 200 chars) that captures:
    1. Genre/style fusion
    2. Key instruments or production techniques  
    3. Mood/energy descriptor
    4. Optional: structural hint (e.g., "builds to drop", "minimal loop")
    
    Output format: "genre, instruments, mood, technique
    '''

In [31]:
generate_suno_prompt(semantic_tags, rhythmic_features)

'\n    You are a music prompt engineer for Suno AI. \n    Given these extracted features from a reference track:\n    - Semantic tags: [(\'lo-fi hip hop\', 0.275363028049469), (\'808 drum machine\', 0.24586784839630127), (\'sidechain compression\', 0.24460382759571075), (\'drum and bass\', 0.15425853431224823), (\'reverb-heavy vocals\', 0.12697862088680267)]\n    - Tempo: 99.38401442307692 BPM\n    - Energy level: 0.25675728917121887/1.0\n    - Spectral character: 1224.1556494277565\n\n    Generate a concise, effective Suno prompt (max 200 chars) that captures:\n    1. Genre/style fusion\n    2. Key instruments or production techniques  \n    3. Mood/energy descriptor\n    4. Optional: structural hint (e.g., "builds to drop", "minimal loop")\n\n    Output format: "genre, instruments, mood, technique\n    '